# Ejercicio 3 — Capital Económico con Cópulas

## Máster Executive en Finanzas Cuantitativas 2026 — AFI Global Education
### Fundamentos Matemáticos: Probabilidad y Simulación

---

### Enunciado

El **capital económico** de un banco es el capital necesario para absorber pérdidas inesperadas con cierto nivel de confianza en un horizonte (un año). Se calcula como un **percentil de la distribución de pérdidas agregadas**. Fijamos el **percentil 95%** y la convención: *un valor positivo es una pérdida*. El banco tiene exposición crediticia en cinco países, con pérdidas modeladas como normales:

| País | $\mu$ | $\sigma$ | | País | $\mu$ | $\sigma$ |
|---|:--:|:--:|---|---|:--:|:--:|
| Brasil | 10 | 0.75 | | Italia | 12 | 2.30 |
| Chile | 15 | 1.00 | | Alemania | 19 | 4.00 |
| Francia | 18 | 3.00 | | | | |

### Hoja de ruta de la resolución

| Paso | Qué se pide | Herramienta |
|:--:|---|---|
| **1** | Capital *stand-alone* por país y su suma | Percentil de una normal |
| **2** | Índice sintético macro por país = 1.ª componente principal | PCA sobre `series_macro.xlsx` |
| **3** | Matriz de correlaciones entre países | Correlación de los índices |
| **4** | Capital diversificado con **cópula Gaussiana** | Normal multivariante (Cholesky) |
| **5** | Capital diversificado con **cópula t-Student** | t multivariante |
| **6** | Discutir diversificación y efecto del tipo de cópula | Comparación + *tail dependence* |

**Idea global.** Sumar los peores casos de cada país por separado (capital *stand-alone*) **sobrestima** el riesgo, porque presupone que todos pierden a la vez. La realidad es que los países están **imperfectamente correlacionados**: ahí nace el *beneficio de diversificación*. Para cuantificarlo necesitamos (i) **estimar la correlación** entre países —vía PCA sobre datos macro— y (ii) **simular** la pérdida conjunta. La forma de "pegar" las marginales con la correlación es una **cópula**, y compararemos dos: la Gaussiana y la t-Student.

---
## Configuración del entorno

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import os

# ── Estilo gráfico coherente ────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

# ── Semilla fija (reproducibilidad) ─────────────────────────────────────────
SEED = 42
rng  = np.random.default_rng(SEED)

# ── Parámetros del problema (del enunciado) ─────────────────────────────────
PAISES    = ["Brazil", "Chile", "France", "Italy", "Germany"]  # nombres en el Excel
PAISES_ES = ["Brasil", "Chile", "Francia", "Italia", "Alemania"]
MU    = np.array([10.0, 15.0, 18.0, 12.0, 19.0])   # medias de pérdida (μ)
SIGMA = np.array([ 0.75, 1.00,  3.00,  2.30,  4.00]) # desviaciones típicas (σ)
CONF  = 0.95                                          # nivel de confianza
N_SIM = 500_000                                       # nº de simulaciones Monte Carlo
NU_T  = 5                                             # grados de libertad de la cópula t

# ── Ruta al Excel (AJUSTAR según dónde esté el notebook) ─────────────────────
EXCEL_PATH = "../../series_macro.xlsx"

os.makedirs("resultados", exist_ok=True)
print(f"Entorno listo · SEED={SEED} · N_SIM={N_SIM:,} · confianza={CONF:.0%} · ν(t)={NU_T}")

---
## Paso 1 — Capital *stand-alone* por país

### Desarrollo

En este primer paso tratamos cada país de forma aislada: $L_i\sim\mathcal N(\mu_i,\sigma_i^2)$. El capital económico al 95% es, por definición, el **percentil 95** de la distribución de pérdidas. Para una normal, ese percentil tiene forma cerrada:

$$CE_i^{\text{SA}}=F_{L_i}^{-1}(0.95)=\mu_i+\sigma_i\,\Phi^{-1}(0.95)=\mu_i+\sigma_i\cdot 1.6449,$$

donde $\Phi^{-1}(0.95)=z_{0.95}\approx1.6449$ es el cuantil 95% de la normal estándar.

El **capital stand-alone total** es la suma directa:

$$CE^{\text{SA}}_{\text{total}}=\sum_{i=1}^{5}CE_i^{\text{SA}}.$$

**Interpretación.** Sumar percentiles equivale a suponer que los cinco países alcanzan su peor escenario **simultáneamente**, es decir, **correlación perfecta** ($\rho=1$). Es el escenario más conservador y servirá como cota superior contra la que medir la diversificación.

In [ ]:
# ── Capital stand-alone: percentil 95% de cada normal ───────────────────────
z95 = stats.norm.ppf(CONF)                 # cuantil 95% de N(0,1)
CE_standalone = MU + SIGMA * z95           # CE_i = μ_i + σ_i·z95
CE_standalone_total = CE_standalone.sum()

print(f"z_{{0.95}} = Φ⁻¹(0.95) = {z95:.6f}\n")
print(f"{'País':<10} {'μ':>6} {'σ':>6} {'CE = μ + σ·z95':>16}")
print("-" * 42)
for p, mu, s, ce in zip(PAISES_ES, MU, SIGMA, CE_standalone):
    print(f"{p:<10} {mu:>6.1f} {s:>6.2f} {ce:>16.4f}")
print("-" * 42)
print(f"{'TOTAL':<10} {'':>6} {'':>6} {CE_standalone_total:>16.4f}")

In [ ]:
# ── Figura 1: capital stand-alone por país ──────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
colores = ["#4e79a7", "#f28e2b", "#e15759", "#76b7b2", "#59a14f"]
bars = ax.bar(PAISES_ES, CE_standalone, color=colores, alpha=0.85, edgecolor="k")
for b, v in zip(bars, CE_standalone):
    ax.text(b.get_x()+b.get_width()/2, v+0.1, f"{v:.2f}", ha="center", fontsize=10)
ax.set_ylabel("capital económico (u.m.)")
ax.set_title(f"Capital stand-alone por país (percentil {CONF:.0%})\n"
             f"Total (sin diversificación) = {CE_standalone_total:.2f}")
fig.tight_layout()
fig.savefig("resultados/grafico_07_capital_standalone.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada en resultados/grafico_07_capital_standalone.png")

---
## Paso 2 — Índice sintético macroeconómico por país (PCA)

### ¿Por qué PCA?

Cada país tiene varias series macro anuales (PIB, inflación, desempleo, tipo de depósito). Queremos resumirlas en **un único índice** que capture el "estado" macroeconómico del país a lo largo del tiempo. El **Análisis de Componentes Principales (PCA)** hace exactamente eso: encuentra la combinación lineal de las series que **maximiza la varianza** explicada. La **primera componente principal (PC1)** es ese índice sintético.

### Procedimiento (por país)

1. **Disponer los datos** como matriz (años × series).
2. **Estandarizar** cada serie (media 0, varianza 1), para que ninguna domine por su escala (p. ej. la inflación de Brasil llegó a >2000%, frente a un desempleo en torno a 10%).
3. **PCA** y quedarse con la PC1: la proyección de cada año sobre la dirección de máxima varianza.

### Nota importante sobre los datos

Al inspeccionar `series_macro.xlsx` se observa que **el número de series no es igual para todos los países**: Brasil y Chile tienen 4 (incluyen *Deposit interest rate*), mientras que Francia, Italia y Alemania tienen 3 (sin esa serie). El código lo detecta automáticamente y aplica PCA a las series **realmente disponibles** de cada país, sin asumir un número fijo. La celda imprime cuántas series usa por país para dejar constancia.

In [ ]:
# ── Lectura del Excel ───────────────────────────────────────────────────────
df_raw = pd.read_excel(EXCEL_PATH, sheet_name="Data")

# Las columnas de años tienen formato '1991 [YR1991]' → las detectamos por dígitos
year_cols = [c for c in df_raw.columns if str(c)[:4].isdigit()]
print(f"Horizonte temporal: {len(year_cols)} años ({year_cols[0][:4]}–{year_cols[-1][:4]})")
print(f"Dimensiones de la hoja: {df_raw.shape}")

In [ ]:
# ── PCA por país: índice sintético = primera componente principal ───────────
pca_indices = {}   # país → serie temporal del índice (1 valor por año)
pca_var_exp = {}   # país → % de varianza explicada por la PC1
n_series    = {}   # país → nº de series macro disponibles

for pais in PAISES:
    # 1) Seleccionar las filas (series) de este país
    sub = df_raw[df_raw["Country Name"] == pais][year_cols]
    n_series[pais] = sub.shape[0]
    # 2) Transponer → (años, series) y estandarizar cada serie
    Xp   = sub.values.T.astype(float)             # (34 años, k series)
    Xstd = StandardScaler().fit_transform(Xp)     # media 0, varianza 1 por serie
    # 3) PCA y extracción de la PC1 (proyección de cada año)
    pca  = PCA(n_components=1, random_state=SEED)
    pca_indices[pais] = pca.fit_transform(Xstd)[:, 0]
    pca_var_exp[pais] = pca.explained_variance_ratio_[0]

print(f"{'País':<10} {'nº series':>10} {'varianza expl. PC1':>20}")
print("-" * 42)
for p_en, p_es in zip(PAISES, PAISES_ES):
    print(f"{p_es:<10} {n_series[p_en]:>10} {pca_var_exp[p_en]:>19.1%}")

In [ ]:
# ── Figura 2: evolución temporal del índice sintético de cada país ──────────
years = [int(c[:4]) for c in year_cols]
fig, axes = plt.subplots(2, 3, figsize=(15, 7.5))
for i, (p_en, p_es) in enumerate(zip(PAISES, PAISES_ES)):
    ax = axes.flatten()[i]
    ax.plot(years, pca_indices[p_en], color=plt.cm.tab10(i), lw=2, marker="o", ms=3)
    ax.axhline(0, color="k", lw=0.8, ls="--")
    ax.set_title(f"{p_es}  ·  PC1 explica {pca_var_exp[p_en]:.1%}\n({n_series[p_en]} series macro)")
    ax.set_xlabel("año"); ax.set_ylabel("índice (PC1)")
axes.flatten()[-1].set_visible(False)   # sólo hay 5 países, ocultamos el 6.º panel
fig.suptitle("Índice sintético macroeconómico por país — 1.ª componente principal (1991–2024)",
             fontsize=13)
fig.tight_layout()
fig.savefig("resultados/grafico_08_indices_pca.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada en resultados/grafico_08_indices_pca.png")

---
## Paso 3 — Matriz de correlaciones entre países

El enunciado pide estimar la correlación entre países como la **correlación entre sus índices sintéticos**. Apilamos los cinco índices (cada uno una serie de 34 años) en una matriz $34\times5$ y calculamos la matriz de correlaciones de Pearson $5\times5$.

Esta matriz $\Sigma$ es el ingrediente que conecta los datos macro históricos con la simulación de pérdidas: dos países cuyas economías se mueven juntas (correlación alta) tenderán a sufrir pérdidas a la vez, reduciendo la diversificación.

**Requisito técnico.** Para poder simular después, $\Sigma$ debe ser **semidefinida positiva** (todos sus valores propios $\ge0$). Lo verificamos con un `assert`, ya que es condición necesaria para la descomposición de Cholesky del Paso 4.

In [ ]:
# ── Matriz de correlaciones entre los índices sintéticos ────────────────────
indices_matrix = np.column_stack([pca_indices[p] for p in PAISES])  # (34, 5)
corr_matrix    = np.corrcoef(indices_matrix.T)                       # (5, 5)

df_corr = pd.DataFrame(corr_matrix, index=PAISES_ES, columns=PAISES_ES)
print("Matriz de correlaciones entre índices sintéticos (PC1):\n")
print(df_corr.round(3).to_string())

# Verificación: la matriz debe ser semidefinida positiva (para Cholesky)
eigvals = np.linalg.eigvalsh(corr_matrix)
print(f"\nValores propios: {eigvals.round(4)}")
assert np.all(eigvals >= -1e-8), "La matriz de correlaciones NO es semidefinida positiva"
print("✓ Matriz semidefinida positiva → válida para simular")

In [ ]:
# ── Figura 3: mapa de calor de la matriz de correlaciones ───────────────────
fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(corr_matrix, cmap="RdYlGn", vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, label="correlación de Pearson")
ax.set_xticks(range(5)); ax.set_xticklabels(PAISES_ES, rotation=30, ha="right")
ax.set_yticks(range(5)); ax.set_yticklabels(PAISES_ES)
for i in range(5):
    for j in range(5):
        val = corr_matrix[i, j]
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                fontsize=10, color="white" if abs(val) > 0.6 else "black")
ax.set_title("Correlación entre índices sintéticos (PC1) · 1991–2024")
fig.tight_layout()
fig.savefig("resultados/grafico_09_heatmap_correlaciones.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada en resultados/grafico_09_heatmap_correlaciones.png")

**Lectura económica de la matriz.** Brasil y Chile están fuertemente correlacionados (≈0.67): dos economías latinoamericanas con ciclos parecidos. Francia y Alemania muestran correlación moderada (≈0.54), coherente con la integración de la zona euro. En cambio, los países latinoamericanos correlacionan **negativamente** con Francia (≈−0.2/−0.24) y casi nada con Alemania (≈0.03): sus ciclos son en gran medida independientes o incluso opuestos. Esta heterogeneidad es justo lo que producirá beneficio de diversificación.

---
## Paso 4 — Capital diversificado con **cópula Gaussiana**

### ¿Qué es una cópula y por qué Gaussiana?

Una **cópula** separa dos cosas: las **distribuciones marginales** (aquí, cada pérdida es $\mathcal N(\mu_i,\sigma_i^2)$) y la **estructura de dependencia** entre ellas. La **cópula Gaussiana** impone que la dependencia tenga forma de normal multivariante con matriz de correlación $\Sigma$.

Como en nuestro caso las marginales **también** son normales, el resultado es simplemente un **vector normal multivariante** $\mathbf L\sim\mathcal N_5(\boldsymbol\mu,\,D\Sigma D)$ con $D=\operatorname{diag}(\sigma_i)$. Esto simplifica mucho la simulación.

### Algoritmo (vía descomposición de Cholesky)

Para simular $\mathbf Z\sim\mathcal N_5(\mathbf 0,\Sigma)$ usamos el hecho de que, si $\Sigma=LL^\top$ (Cholesky) y $\mathbf z\sim\mathcal N_5(\mathbf 0,I)$ son normales independientes, entonces $L\mathbf z$ tiene exactamente covarianza $\Sigma$:

$$\operatorname{Cov}(L\mathbf z)=L\,\operatorname{Cov}(\mathbf z)\,L^\top=L\,I\,L^\top=LL^\top=\Sigma.$$

Pasos:
1. $\Sigma=LL^\top$ (Cholesky).
2. Simular $\mathbf z\sim\mathcal N_5(\mathbf 0,I)$ ($N$ veces) y formar $\mathbf Z=\mathbf z L^\top$.
3. Aplicar marginales: como son normales, $L_i=\mu_i+\sigma_i Z_i$.
4. Pérdida total $L_{\text{tot}}=\sum_i L_i$ y capital $CE^{\text{G}}=\text{percentil}_{95\%}(L_{\text{tot}})$.

In [ ]:
# ── Simulación con cópula Gaussiana ─────────────────────────────────────────
# 1) Descomposición de Cholesky de la matriz de correlaciones
L_chol = np.linalg.cholesky(corr_matrix)            # Σ = L·Lᵀ

# 2) Simular N vectores normales correlacionados Z ~ N(0, Σ)
Z = rng.standard_normal((N_SIM, 5)) @ L_chol.T      # (N_SIM, 5)

# 3) Aplicar marginales normales: L_i = μ_i + σ_i·Z_i
L_gauss = MU + SIGMA * Z                            # (N_SIM, 5)

# 4) Pérdida total del portafolio y capital diversificado
L_total_gauss = L_gauss.sum(axis=1)
CE_div_gauss  = np.percentile(L_total_gauss, 95)

print(f"Cópula GAUSSIANA")
print(f"  pérdida total media : {L_total_gauss.mean():.4f}   (teórica Σμ = {MU.sum():.1f})")
print(f"  pérdida total std   : {L_total_gauss.std():.4f}")
print(f"  capital diversif.   : {CE_div_gauss:.4f}")

# Verificación: las correlaciones simuladas reproducen Σ
max_err = np.abs(np.corrcoef(L_gauss.T) - corr_matrix).max()
print(f"  max|ρ̂ − Σ|         : {max_err:.5f}   (debe ser ≈0)")
assert max_err < 0.02
print("  ✓ la simulación reproduce la matriz de correlaciones")

---
## Paso 5 — Capital diversificado con **cópula t-Student**

### Motivación: dependencia en las colas

La cópula Gaussiana tiene una limitación conocida: **no captura la dependencia en las colas** (*tail dependence*). En la práctica, en una crisis los países tienden a desplomarse **a la vez** mucho más de lo que predice una normal. La **cópula t-Student**, con sus colas más pesadas, modela ese fenómeno: añade un factor común $\chi^2$ que, cuando toma valores pequeños, **amplifica simultáneamente** todas las pérdidas.

### Algoritmo

Una t multivariante se construye dividiendo una normal multivariante por un factor $\chi^2$ común:

1. $\mathbf Z\sim\mathcal N_5(\mathbf 0,\Sigma)$ (igual que antes).
2. $W\sim\chi^2_\nu$ **independiente**, con $\nu=5$ grados de libertad.
3. $\mathbf T=\mathbf Z/\sqrt{W/\nu}$: vector con distribución **t multivariante** ($\nu$ g.l., correlación $\Sigma$). El factor $W$ es **compartido** por las 5 componentes → ahí está la dependencia de cola.
4. Pasar a uniformes con la CDF t: $U_i=F_{t_\nu}(T_i)$.
5. Aplicar las marginales **normales** (el enunciado mantiene las pérdidas normales): $L_i=\mu_i+\sigma_i\,\Phi^{-1}(U_i)$.
6. Capital $CE^{\text t}=\text{percentil}_{95\%}(\sum_i L_i)$.

> **Matiz importante.** Cambiamos la **cópula** (cómo se relacionan los países), no las **marginales** (cada pérdida sigue siendo $\mathcal N(\mu_i,\sigma_i^2)$). Por eso el paso 5 transforma de la t a la normal: aplicamos $F_{t_\nu}$ y luego $\Phi^{-1}$ para conservar marginales normales con dependencia tipo-t.

In [ ]:
# ── Simulación con cópula t-Student ─────────────────────────────────────────
rng_t = np.random.default_rng(SEED + 10)   # semilla separada → independencia de la Gaussiana

# 1) Z ~ N(0, Σ) correlacionada (mismo Cholesky)
Z_t  = rng_t.standard_normal((N_SIM, 5)) @ L_chol.T
# 2) W ~ χ²_ν independiente
W    = rng_t.chisquare(NU_T, size=N_SIM)
# 3) T = Z / sqrt(W/ν) → t multivariante (el factor W es COMÚN a las 5 componentes)
T    = Z_t / np.sqrt(W / NU_T)[:, None]
# 4) pasar a uniformes con la CDF de la t_ν
U    = stats.t.cdf(T, df=NU_T)
# 5) aplicar marginales NORMALES: L_i = μ_i + σ_i·Φ⁻¹(U_i)
L_t  = MU + SIGMA * stats.norm.ppf(U)
# 6) pérdida total y capital
L_total_t = L_t.sum(axis=1)
CE_div_t  = np.percentile(L_total_t, 95)

print(f"Cópula t-STUDENT (ν={NU_T})")
print(f"  pérdida total media : {L_total_t.mean():.4f}   (teórica Σμ = {MU.sum():.1f})")
print(f"  pérdida total std   : {L_total_t.std():.4f}")
print(f"  capital diversif.   : {CE_div_t:.4f}")

---
## Paso 6 — Comparación y discusión

Reunimos los tres capitales y cuantificamos el **beneficio de diversificación** (cuánto capital se ahorra frente al stand-alone) y el **efecto del tipo de cópula**.

In [ ]:
# ── Resumen comparativo de capitales ────────────────────────────────────────
ben_g = CE_standalone_total - CE_div_gauss      # ahorro Gaussiana
ben_t = CE_standalone_total - CE_div_t          # ahorro t-Student
pct_g = ben_g / CE_standalone_total * 100
pct_t = ben_t / CE_standalone_total * 100

print("═" * 58)
print(f"  Capital stand-alone (sin diversificar) : {CE_standalone_total:>9.4f}")
print(f"  Capital diversificado · Gaussiana      : {CE_div_gauss:>9.4f}")
print(f"  Capital diversificado · t-Student      : {CE_div_t:>9.4f}")
print("─" * 58)
print(f"  Beneficio diversificación · Gaussiana  : {ben_g:>9.4f}  ({pct_g:.1f}%)")
print(f"  Beneficio diversificación · t-Student  : {ben_t:>9.4f}  ({pct_t:.1f}%)")
print(f"  Diferencia (t − Gaussiana)             : {CE_div_t - CE_div_gauss:>+9.4f}")
print("═" * 58)

In [ ]:
# ── Evidencia de tail dependence: ¿con qué frecuencia pierden TODOS a la vez? ─
# Contamos escenarios donde los 5 países superan simultáneamente μ+2σ (cola).
umbral = MU + 2 * SIGMA
p_all_gauss = np.mean(np.all(L_gauss > umbral, axis=1))
p_all_t     = np.mean(np.all(L_t     > umbral, axis=1))

print("Probabilidad de que los 5 países superen μ+2σ SIMULTÁNEAMENTE:")
print(f"  cópula Gaussiana : {p_all_gauss:.6f}")
print(f"  cópula t-Student : {p_all_t:.6f}")
print(f"  → con la cópula t es ~{p_all_t/max(p_all_gauss,1e-9):.0f}× más probable")
print("\nEsta es la 'tail dependence': la t concentra mucho más riesgo de")
print("pérdidas conjuntas extremas, aunque el percentil 95% global salga similar.")

In [ ]:
# ── Figura 4: distribuciones de pérdida total y comparación de capitales ────
fig, (axL, axR) = plt.subplots(1, 2, figsize=(14, 5))

# (izq.) histogramas de la pérdida total bajo ambas cópulas
rango = (min(L_total_gauss.min(), L_total_t.min()),
         max(L_total_gauss.max(), L_total_t.max()))
axL.hist(L_total_gauss, bins=160, range=rango, density=True, alpha=0.5,
         color="steelblue", label=f"Gaussiana (CE={CE_div_gauss:.2f})")
axL.hist(L_total_t, bins=160, range=rango, density=True, alpha=0.5,
         color="orange", label=f"t-Student (CE={CE_div_t:.2f})")
axL.axvline(CE_div_gauss, color="steelblue", lw=2, ls="--")
axL.axvline(CE_div_t,     color="darkorange", lw=2, ls="--")
axL.axvline(CE_standalone_total, color="red", lw=2, ls="-.",
            label=f"stand-alone ({CE_standalone_total:.2f})")
axL.set_xlabel("pérdida total (u.m.)"); axL.set_ylabel("densidad")
axL.set_title("Distribución de pérdidas agregadas")
axL.legend(fontsize=9)

# (der.) comparación de los tres capitales con el ahorro de diversificación
etiq = ["stand-alone", "Gaussiana", f"t-Student\n(ν={NU_T})"]
vals = [CE_standalone_total, CE_div_gauss, CE_div_t]
cols = ["#e15759", "#4e79a7", "#f28e2b"]
bars = axR.bar(etiq, vals, color=cols, alpha=0.85, edgecolor="k", width=0.45)
for b, v in zip(bars, vals):
    axR.text(b.get_x()+b.get_width()/2, v+0.1, f"{v:.2f}",
             ha="center", fontsize=11, fontweight="bold")
# flecha del beneficio de diversificación
axR.annotate("", xy=(1, CE_div_gauss), xytext=(0, CE_standalone_total),
             arrowprops=dict(arrowstyle="<->", color="green", lw=2))
axR.text(0.5, (CE_standalone_total+CE_div_gauss)/2 + 0.5,
         f"−{ben_g:.1f}\n({pct_g:.0f}%)", ha="center", color="green", fontsize=10)
axR.set_ylabel("capital económico (u.m.)")
axR.set_ylim(0, CE_standalone_total*1.12)
axR.set_title("Capital: stand-alone vs diversificado (p95)")

fig.suptitle("Paso 6 — diversificación y efecto del tipo de cópula", fontsize=13)
fig.tight_layout()
fig.savefig("resultados/grafico_10_comparacion_capitales.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada en resultados/grafico_10_comparacion_capitales.png")

---
## Conclusiones

Tablas-resumen generadas **desde las variables del notebook**.

In [ ]:
from IPython.display import display, HTML

# Tabla 1: capital stand-alone por país
filas1 = "".join(
    f"<tr><td style='padding:5px 10px'>{p}</td>"
    f"<td style='padding:5px 10px;text-align:right'>{mu:.1f}</td>"
    f"<td style='padding:5px 10px;text-align:right'>{s:.2f}</td>"
    f"<td style='padding:5px 10px;text-align:right'>{ce:.4f}</td></tr>"
    for p, mu, s, ce in zip(PAISES_ES, MU, SIGMA, CE_standalone))

# Tabla 2: comparación de capitales
filas2 = (
    f"<tr><td style='padding:5px 10px'>stand-alone (sin corr.)</td>"
    f"<td style='padding:5px 10px;text-align:right'>{CE_standalone_total:.4f}</td>"
    f"<td style='padding:5px 10px;text-align:right'>—</td>"
    f"<td style='padding:5px 10px;text-align:right'>—</td></tr>"
    f"<tr><td style='padding:5px 10px'>cópula Gaussiana</td>"
    f"<td style='padding:5px 10px;text-align:right'>{CE_div_gauss:.4f}</td>"
    f"<td style='padding:5px 10px;text-align:right'>{ben_g:.4f}</td>"
    f"<td style='padding:5px 10px;text-align:right'>{pct_g:.1f}%</td></tr>"
    f"<tr style='background:#fee'><td style='padding:5px 10px'>cópula t-Student (ν={NU_T})</td>"
    f"<td style='padding:5px 10px;text-align:right'>{CE_div_t:.4f}</td>"
    f"<td style='padding:5px 10px;text-align:right'>{ben_t:.4f}</td>"
    f"<td style='padding:5px 10px;text-align:right'>{pct_t:.1f}%</td></tr>")

html = f"""
<h4 style='margin-bottom:4px'>Tabla 1 · Capital stand-alone por país</h4>
<table style='border-collapse:collapse;font-size:12px'>
<tr style='background:#264653;color:white;font-weight:bold'>
<td style='padding:5px 10px'>País</td><td style='padding:5px 10px'>μ</td>
<td style='padding:5px 10px'>σ</td><td style='padding:5px 10px'>CE (p95)</td></tr>
{filas1}
<tr style='background:#e9c46a;font-weight:bold'><td style='padding:5px 10px'>TOTAL</td>
<td></td><td></td><td style='padding:5px 10px;text-align:right'>{CE_standalone_total:.4f}</td></tr>
</table>
<br>
<h4 style='margin-bottom:4px'>Tabla 2 · Comparación de capitales diversificados</h4>
<table style='border-collapse:collapse;font-size:12px'>
<tr style='background:#264653;color:white;font-weight:bold'>
<td style='padding:5px 10px'>Método</td><td style='padding:5px 10px'>Capital (p95)</td>
<td style='padding:5px 10px'>Ahorro</td><td style='padding:5px 10px'>% ahorro</td></tr>
{filas2}
</table>
"""
display(HTML(html))

### Discusión final

**Impacto de la diversificación.** El capital stand-alone (**92.18**) supone que los cinco países alcanzan su peor escenario a la vez. Al introducir la correlación real estimada por PCA, el capital diversificado baja a **~86**, un **ahorro del ~6.7%**. Ese ahorro es exactamente el *beneficio de diversificación*: como las economías no están perfectamente correlacionadas (e incluso algunas correlacionan negativamente, como Brasil/Chile frente a Francia), es improbable que todas pierdan el máximo simultáneamente, y el banco necesita menos capital.

**Efecto del tipo de cópula.** Los capitales Gaussiano (**86.03**) y t-Student (**85.93**) salen muy parecidos en el percentil 95%. Esto **no** significa que las cópulas sean equivalentes: como muestra el análisis de *tail dependence*, la probabilidad de que **los cinco países sufran pérdidas extremas a la vez** ($>\mu+2\sigma$) es **~50× mayor** con la cópula t. Lo que ocurre es que ese riesgo de cola conjunta vive **más allá** del percentil 95% (en el 99% o el 99.9%), donde la t se separaría claramente de la Gaussiana. Con correlaciones moderadas como las estimadas aquí, al 95% ambas casi coinciden.

**Conclusión práctica.** La elección del modelo de dependencia importa, y mucho, cuanto **más extrema** sea la confianza exigida. La cópula Gaussiana es adecuada para niveles moderados, pero **subestima el riesgo sistémico** en las colas; por eso la regulación (Basilea III/IV) y la gestión prudente del riesgo favorecen cópulas con dependencia de cola (t-Student) y niveles de confianza altos (99.9%), donde la diferencia se vuelve material. El parámetro $\nu$ controla la intensidad del efecto: a menor $\nu$, colas más pesadas y mayor capital.